# FordRetain ML Workflow\n\nEste notebook demonstra como utilizar clustering para identificar perfis de clientes (Etapa 1) e como treinar um classificador para prever o perfil de novos clientes (Etapa 2). Use dados sintéticos (base1.csv e base2.csv) fornecidos no diretório datasets como exemplo. Adapte com os dados reais do professor ao treinar o modelo final.\n

In [ ]:
import pandas as pd\nfrom sklearn.preprocessing import OneHotEncoder\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.cluster import KMeans\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, f1_score, confusion_matrix\n\n# Carregue a base completa de clientes (Base 1) com a coluna 'profile'\nbase1 = pd.read_csv('datasets/base1.csv')\n\n# Selecione apenas as colunas de entrada para o clustering\nX_cluster = base1[['age', 'region', 'car_model', 'payment_type', 'purchase_channel', 'brand_history']]\n\n# Pré-processamento: codificação one-hot para colunas categóricas\ncategorical_cols = ['region', 'car_model', 'payment_type', 'purchase_channel', 'brand_history']\nnumeric_cols = ['age']\npreprocess = ColumnTransformer(transformers=[\n    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_cols),\n    ('numeric', 'passthrough', numeric_cols)\n])\n\n# Pipeline de clustering com K-Means\ncluster_model = Pipeline(steps=[\n    ('preprocess', preprocess),\n    ('cluster', KMeans(n_clusters=4, random_state=42))\n])\n\ncluster_labels = cluster_model.fit_predict(X_cluster)\nbase1['cluster'] = cluster_labels\n\n# Atribuição de nomes de perfil com base na análise de clusters (substitua pela lógica real)\nprofile_mapping = {\n    0: 'Cliente Fiel',\n    1: 'Cliente Economico',\n    2: 'Cliente Esquecido',\n    3: 'Cliente de Abandono'\n}\nbase1['derived_profile'] = base1['cluster'].map(profile_mapping)\n\n# Agora treine um classificador usando apenas as colunas disponíveis no momento da compra\nfeatures = X_cluster.copy()\ntarget = base1['profile']  # coluna de perfil real fornecida pelo professor\n\nX_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)\n\n# O pipeline de classificação reutiliza o preprocess e adiciona um RandomForest\nclf = Pipeline(steps=[\n    ('preprocess', preprocess),\n    ('model', RandomForestClassifier(n_estimators=200, random_state=42))\n])\n\nclf.fit(X_train, y_train)\ny_pred = clf.predict(X_test)\nprint('Accuracy:', accuracy_score(y_test, y_pred))\nprint('F1-score:', f1_score(y_test, y_pred, average='weighted'))\nprint('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))\n\n# Salve o modelo para uso na API (por exemplo, usando joblib)\nimport joblib\njoblib.dump(clf, 'models/profile_classifier.joblib')\n\n# Prever o perfil de novos clientes (Base 2)\nbase2 = pd.read_csv('datasets/base2.csv')\npredictions = clf.predict(base2[['age', 'region', 'car_model', 'payment_type', 'purchase_channel', 'brand_history']])\nbase2['predicted_profile'] = predictions\nbase2.to_csv('datasets/base2_predictions.csv', index=False)\n